In [2]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.power import NormalIndPower, TTestIndPower

np.random.seed(42)

pd.set_option('display.float_format', '{:.4f}'.format)

print('All libraries loaded successfully')

All libraries loaded successfully


In [3]:
p1_email = 0.22
mde_email = 0.03
p2_email = p1_email + mde_email

alpha = 0.05
power = 0.80

effect_size = 2 * (np.arcsin(np.sqrt(p2_email)) - np.arcsin(np.sqrt(p1_email)))

power_analysis = NormalIndPower()
n_per_group_email = power_analysis.solve_power(
    effect_size=effect_size,
    alpha=alpha,
    power=power,
    alternative='two-sided'
)
n_per_group_email = int(np.ceil(n_per_group_email))

print(f'Baseline open rate:            {p1_email:.2%}')
print(f'Target open rate:              {p2_email:.2%}')
print(f'Minimum Detectable Effect:     +{mde_email:.2%}')
print(f'Alpha:                         {alpha}')
print(f'Power:                         {power:.0%}')
print(f'Effect size (Cohen\'s h):       {effect_size:.4f}')
print(f'Required n per group:          {n_per_group_email:,}')
print(f'Total emails needed:           {n_per_group_email * 2:,}')

Baseline open rate:            22.00%
Target open rate:              25.00%
Minimum Detectable Effect:     +3.00%
Alpha:                         0.05
Power:                         80%
Effect size (Cohen's h):       0.0708
Required n per group:          3,133
Total emails needed:           6,266


In [4]:
n_email = int(n_per_group_email * 1.5)

true_control_rate = 0.220
true_treatment_rate = 0.265

control_opens = np.random.binomial(1, true_control_rate, n_email)
treatment_opens = np.random.binomial(1, true_treatment_rate, n_email)

df_email = pd.DataFrame({
    'customer_id': [f'C{i:05d}' for i in range(n_email * 2)],
    'group': ['control'] * n_email + ['treatment'] * n_email,
    'opened': list(control_opens) + list(treatment_opens)
})

summary = df_email.groupby('group')['opened'].agg(['sum', 'count', 'mean'])
summary.columns = ['opens', 'sent', 'open_rate']
print(summary.round(4))

           opens  sent  open_rate
group                            
control     1026  4699     0.2183
treatment   1185  4699     0.2522


In [5]:
n_control = summary.loc['control', 'sent']
n_treatment = summary.loc['treatment', 'sent']
opens_control = summary.loc['control', 'opens']
opens_treatment = summary.loc['treatment', 'opens']
rate_control = summary.loc['control', 'open_rate']
rate_treatment = summary.loc['treatment', 'open_rate']

z_stat_email, p_value_email = proportions_ztest(
    count=np.array([opens_treatment, opens_control]),
    nobs=np.array([n_treatment, n_control]),
    alternative='two-sided'
)

z_critical = stats.norm.ppf(1 - alpha/2)

se_diff = np.sqrt(
    (rate_control * (1 - rate_control) / n_control) +
    (rate_treatment * (1 - rate_treatment) / n_treatment)
)

diff = rate_treatment - rate_control
ci_low_email = diff - z_critical * se_diff
ci_high_email = diff + z_critical * se_diff

print(f'Control open rate:    {rate_control:.4f}')
print(f'Treatment open rate:  {rate_treatment:.4f}')
print(f'Lift:                 {diff*100:+.2f} percentage points')
print(f'Z-statistic:          {z_stat_email:.4f}')
print(f'P-value:              {p_value_email:.6f}')
print(f'95% CI:               [{ci_low_email*100:.2f}%, {ci_high_email*100:.2f}%]')
print(f'Significant:          {p_value_email < alpha}')

Control open rate:    0.2183
Treatment open rate:  0.2522
Lift:                 +3.38 percentage points
Z-statistic:          3.8668
P-value:              0.000110
95% CI:               [1.67%, 5.10%]
Significant:          True


In [6]:
emails_sent_per_week = 50000
open_to_click_rate = 0.25
click_to_convert_rate = 0.15
avg_transaction_value = 285.0

control_opens_weekly = emails_sent_per_week * rate_control
control_clicks_weekly = control_opens_weekly * open_to_click_rate
control_converts_weekly = control_clicks_weekly * click_to_convert_rate
control_revenue_weekly = control_converts_weekly * avg_transaction_value

treatment_opens_weekly = emails_sent_per_week * rate_treatment
treatment_clicks_weekly = treatment_opens_weekly * open_to_click_rate
treatment_converts_weekly = treatment_clicks_weekly * click_to_convert_rate
treatment_revenue_weekly = treatment_converts_weekly * avg_transaction_value

incremental_revenue_weekly = treatment_revenue_weekly - control_revenue_weekly
incremental_revenue_annual = incremental_revenue_weekly * 52

print(f'Weekly emails sent:              {emails_sent_per_week:,}')
print()
print(f'Control weekly opens:            {control_opens_weekly:,.0f}')
print(f'Control weekly conversions:      {control_converts_weekly:,.0f}')
print(f'Control weekly revenue:          ${control_revenue_weekly:,.0f}')
print()
print(f'Treatment weekly opens:          {treatment_opens_weekly:,.0f}')
print(f'Treatment weekly conversions:    {treatment_converts_weekly:,.0f}')
print(f'Treatment weekly revenue:        ${treatment_revenue_weekly:,.0f}')
print()
print(f'Incremental revenue (weekly):    +${incremental_revenue_weekly:,.0f}')
print(f'Incremental revenue (annual):    +${incremental_revenue_annual:,.0f}')

Weekly emails sent:              50,000

Control weekly opens:            10,917
Control weekly conversions:      409
Control weekly revenue:          $116,678

Treatment weekly opens:          12,609
Treatment weekly conversions:    473
Treatment weekly revenue:        $134,759

Incremental revenue (weekly):    +$18,082
Incremental revenue (annual):    +$940,245


In [7]:
p1_lp = 0.042
mde_lp = 0.008
p2_lp = p1_lp + mde_lp

effect_size_lp = 2 * (np.arcsin(np.sqrt(p2_lp)) - np.arcsin(np.sqrt(p1_lp)))

n_per_group_lp = power_analysis.solve_power(
    effect_size=effect_size_lp,
    alpha=alpha,
    power=power,
    alternative='two-sided'
)
n_per_group_lp = int(np.ceil(n_per_group_lp))

print(f'Baseline conversion rate:      {p1_lp:.2%}')
print(f'Target conversion rate:        {p2_lp:.2%}')
print(f'Minimum Detectable Effect:     +{mde_lp:.2%}')
print(f'Effect size (Cohen\'s h):       {effect_size_lp:.4f}')
print(f'Required n per group:          {n_per_group_lp:,}')
print(f'Total visitors needed:         {n_per_group_lp * 2:,}')

Baseline conversion rate:      4.20%
Target conversion rate:        5.00%
Minimum Detectable Effect:     +0.80%
Effect size (Cohen's h):       0.0382
Required n per group:          10,744
Total visitors needed:         21,488


In [8]:
n_lp = int(n_per_group_lp * 1.2)

true_control_lp = 0.042
true_treatment_lp = 0.055

lp_control_converts = np.random.binomial(n_lp, true_control_lp)
lp_treatment_converts = np.random.binomial(n_lp, true_treatment_lp)

rate_control_lp = lp_control_converts / n_lp
rate_treatment_lp = lp_treatment_converts / n_lp

print(f'Sample size per group:         {n_lp:,}')
print(f'Control conversions:           {lp_control_converts:,} / {n_lp:,} visitors')
print(f'Treatment conversions:         {lp_treatment_converts:,} / {n_lp:,} visitors')
print(f'Control conversion rate:       {rate_control_lp:.4f}')
print(f'Treatment conversion rate:     {rate_treatment_lp:.4f}')

Sample size per group:         12,892
Control conversions:           572 / 12,892 visitors
Treatment conversions:         744 / 12,892 visitors
Control conversion rate:       0.0444
Treatment conversion rate:     0.0577


In [9]:
z_stat_lp, p_value_lp = proportions_ztest(
    count=np.array([lp_treatment_converts, lp_control_converts]),
    nobs=np.array([n_lp, n_lp]),
    alternative='two-sided'
)

se_diff_lp = np.sqrt(
    (rate_control_lp * (1 - rate_control_lp) / n_lp) +
    (rate_treatment_lp * (1 - rate_treatment_lp) / n_lp)
)

diff_lp = rate_treatment_lp - rate_control_lp
ci_low_lp = diff_lp - z_critical * se_diff_lp
ci_high_lp = diff_lp + z_critical * se_diff_lp

print(f'Control conversion rate:       {rate_control_lp:.4f}')
print(f'Treatment conversion rate:     {rate_treatment_lp:.4f}')
print(f'Lift:                          {diff_lp*100:+.2f} percentage points')
print(f'Relative lift:                 {(rate_treatment_lp/rate_control_lp - 1)*100:+.1f}%')
print(f'Z-statistic:                   {z_stat_lp:.4f}')
print(f'P-value:                       {p_value_lp:.6f}')
print(f'95% CI:                        [{ci_low_lp*100:.2f}%, {ci_high_lp*100:.2f}%]')
print(f'Significant:                   {p_value_lp < alpha}')

Control conversion rate:       0.0444
Treatment conversion rate:     0.0577
Lift:                          +1.33 percentage points
Relative lift:                 +30.1%
Z-statistic:                   4.8672
P-value:                       0.000001
95% CI:                        [0.80%, 1.87%]
Significant:                   True


In [10]:
monthly_visitors = 200000
avg_loan_amount = 8500
interest_rate = 0.119
loan_term_years = 3
approval_rate = 0.62

revenue_per_loan = avg_loan_amount * interest_rate * loan_term_years

control_monthly_apps = monthly_visitors * rate_control_lp
treatment_monthly_apps = monthly_visitors * rate_treatment_lp
incremental_apps = treatment_monthly_apps - control_monthly_apps
incremental_funded_loans = incremental_apps * approval_rate
incremental_revenue_monthly = incremental_funded_loans * revenue_per_loan
incremental_revenue_annual_lp = incremental_revenue_monthly * 12

print(f'Monthly visitors:                  {monthly_visitors:,}')
print(f'Control monthly applications:      {control_monthly_apps:,.0f}')
print(f'Treatment monthly applications:    {treatment_monthly_apps:,.0f}')
print(f'Incremental applications/month:    +{incremental_apps:,.0f}')
print(f'Approval rate:                     {approval_rate:.0%}')
print(f'Incremental funded loans/month:    +{incremental_funded_loans:,.0f}')
print(f'Revenue per funded loan:           ${revenue_per_loan:,.0f}')
print()
print(f'Incremental revenue (monthly):     +${incremental_revenue_monthly:,.0f}')
print(f'Incremental revenue (annual):      +${incremental_revenue_annual_lp:,.0f}')

Monthly visitors:                  200,000
Control monthly applications:      8,874
Treatment monthly applications:    11,542
Incremental applications/month:    +2,668
Approval rate:                     62%
Incremental funded loans/month:    +1,654
Revenue per funded loan:           $3,034

Incremental revenue (monthly):     +$5,020,153
Incremental revenue (annual):      +$60,241,839


In [11]:
mean_control_loan = 9200
std_loan = 3100
mde_loan_amount = 700
mean_treatment_loan = mean_control_loan + mde_loan_amount

cohens_d = mde_loan_amount / std_loan

t_power_analysis = TTestIndPower()
n_per_group_loan = t_power_analysis.solve_power(
    effect_size=cohens_d,
    alpha=alpha,
    power=power,
    alternative='two-sided'
)
n_per_group_loan = int(np.ceil(n_per_group_loan))

print(f'Baseline avg loan amount:      ${mean_control_loan:,}')
print(f'Historical std deviation:      ${std_loan:,}')
print(f'Minimum Detectable Effect:     +${mde_loan_amount:,}')
print(f'Target avg loan amount:        ${mean_treatment_loan:,}')
print(f'Cohen\'s d:                     {cohens_d:.4f}')
print(f'Required n per group:          {n_per_group_loan:,}')
print(f'Total customers needed:        {n_per_group_loan * 2:,}')

Baseline avg loan amount:      $9,200
Historical std deviation:      $3,100
Minimum Detectable Effect:     +$700
Target avg loan amount:        $9,900
Cohen's d:                     0.2258
Required n per group:          309
Total customers needed:        618


In [12]:
n_loan = int(n_per_group_loan * 1.3)

control_loans = np.clip(
    np.random.normal(9200, 3100, n_loan),
    1000, 50000
)
treatment_loans = np.clip(
    np.random.normal(10100, 3400, n_loan),
    1000, 50000
)

print(f'Sample size per group:         {n_loan:,}')
print(f'Control mean:                  ${control_loans.mean():,.2f}')
print(f'Control std:                   ${control_loans.std():,.2f}')
print(f'Control min:                   ${control_loans.min():,.2f}')
print(f'Control max:                   ${control_loans.max():,.2f}')
print()
print(f'Treatment mean:                ${treatment_loans.mean():,.2f}')
print(f'Treatment std:                 ${treatment_loans.std():,.2f}')
print(f'Treatment min:                 ${treatment_loans.min():,.2f}')
print(f'Treatment max:                 ${treatment_loans.max():,.2f}')

Sample size per group:         401
Control mean:                  $9,121.58
Control std:                   $3,168.50
Control min:                   $1,000.00
Control max:                   $17,855.85

Treatment mean:                $10,154.93
Treatment std:                 $3,513.81
Treatment min:                 $1,000.00
Treatment max:                 $20,275.56


In [13]:
t_stat, p_value_loan = stats.ttest_ind(
    treatment_loans,
    control_loans,
    equal_var=False
)

mean_diff = treatment_loans.mean() - control_loans.mean()
se_diff_loan = np.sqrt(
    control_loans.var(ddof=1) / n_loan +
    treatment_loans.var(ddof=1) / n_loan
)

ci_low_loan = mean_diff - z_critical * se_diff_loan
ci_high_loan = mean_diff + z_critical * se_diff_loan

lift_pct_loan = mean_diff / control_loans.mean() * 100

print(f'Control mean loan amount:      ${control_loans.mean():,.2f}')
print(f'Treatment mean loan amount:    ${treatment_loans.mean():,.2f}')
print(f'Difference:                    +${mean_diff:,.2f}')
print(f'Relative lift:                 {lift_pct_loan:+.2f}%')
print(f'T-statistic:                   {t_stat:.4f}')
print(f'P-value:                       {p_value_loan:.6f}')
print(f'95% CI:                        [${ci_low_loan:,.2f}, ${ci_high_loan:,.2f}]')
print(f'Significant:                   {p_value_loan < alpha}')

Control mean loan amount:      $9,121.58
Treatment mean loan amount:    $10,154.93
Difference:                    +$1,033.35
Relative lift:                 +11.33%
T-statistic:                   4.3681
P-value:                       0.000014
95% CI:                        [$569.68, $1,497.02]
Significant:                   True


In [14]:
monthly_loan_offers = 8000
acceptance_rate = 0.38
funding_rate = 0.62
annual_interest_rate = 0.119
avg_loan_term_years = 3

monthly_funded = monthly_loan_offers * acceptance_rate * funding_rate

revenue_per_loan_control = control_loans.mean() * annual_interest_rate * avg_loan_term_years
revenue_per_loan_treatment = treatment_loans.mean() * annual_interest_rate * avg_loan_term_years

incremental_revenue_per_loan = revenue_per_loan_treatment - revenue_per_loan_control
incremental_revenue_monthly = incremental_revenue_per_loan * monthly_funded
incremental_revenue_annual = incremental_revenue_monthly * 12

print(f'Monthly offers sent:               {monthly_loan_offers:,}')
print(f'Acceptance rate:                   {acceptance_rate:.0%}')
print(f'Funding rate:                      {funding_rate:.0%}')
print(f'Monthly funded loans:              {monthly_funded:,.0f}')
print()
print(f'Revenue per loan (control):        ${revenue_per_loan_control:,.0f}')
print(f'Revenue per loan (treatment):      ${revenue_per_loan_treatment:,.0f}')
print(f'Incremental revenue per loan:      +${incremental_revenue_per_loan:,.0f}')
print()
print(f'Incremental revenue (monthly):     +${incremental_revenue_monthly:,.0f}')
print(f'Incremental revenue (annual):      +${incremental_revenue_annual:,.0f}')

Monthly offers sent:               8,000
Acceptance rate:                   38%
Funding rate:                      62%
Monthly funded loans:              1,885

Revenue per loan (control):        $3,256
Revenue per loan (treatment):      $3,625
Incremental revenue per loan:      +$369

Incremental revenue (monthly):     +$695,315
Incremental revenue (annual):      +$8,343,778


In [16]:
summary_results = pd.DataFrame([
    {
        'Test': 'Email Subject Line',
        'Method': 'Z-test (proportions)',
        'Control': f'{rate_control:.2%}',
        'Treatment': f'{rate_treatment:.2%}',
        'Lift': f'+{(rate_treatment/rate_control - 1)*100:.1f}%',
        'P-value': f'{p_value_email:.6f}',
        'Significant': 'Yes',
        'Annual Revenue Impact': f'${incremental_revenue_weekly * 52:,.0f}'
    },
    {
        'Test': 'Landing Page',
        'Method': 'Z-test (proportions)',
        'Control': f'{rate_control_lp:.2%}',
        'Treatment': f'{rate_treatment_lp:.2%}',
        'Lift': f'+{(rate_treatment_lp/rate_control_lp - 1)*100:.1f}%',
        'P-value': f'{p_value_lp:.6f}',
        'Significant': 'Yes',
        'Annual Revenue Impact': f'${incremental_revenue_annual_lp:,.0f}'
    },
    {
        'Test': 'Loan Offer Type',
        'Method': 'T-test (means)',
        'Control': f'${control_loans.mean():,.0f}',
        'Treatment': f'${treatment_loans.mean():,.0f}',
        'Lift': f'+{lift_pct_loan:.1f}%',
        'P-value': f'{p_value_loan:.6f}',
        'Significant': 'Yes',
        'Annual Revenue Impact': f'${incremental_revenue_annual:,.0f}'
    }
])

print('A/B TEST RESULTS SUMMARY')
print('=' * 80)
print(summary_results.to_string(index=False))
print()
print('All three tests statistically significant at alpha = 0.05')
print('Recommendation: Ship all three treatments immediately.')

A/B TEST RESULTS SUMMARY
              Test               Method Control Treatment   Lift  P-value Significant Annual Revenue Impact
Email Subject Line Z-test (proportions)  21.83%    25.22% +15.5% 0.000110         Yes              $940,245
      Landing Page Z-test (proportions)   4.44%     5.77% +30.1% 0.000001         Yes           $60,241,839
   Loan Offer Type       T-test (means)  $9,122   $10,155 +11.3% 0.000014         Yes            $8,343,778

All three tests statistically significant at alpha = 0.05
Recommendation: Ship all three treatments immediately.
